# 🔍 Mapas y Tablas Hash - Mapas Ordenados, Skip Lists y Conjuntos

## Programación III - Lic. en Sistemas

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap10/03_SortedMaps_SkipLists_Sets_Teoria.ipynb)

### 🎯 Objetivos

Al finalizar este notebook el estudiante podrá:
- Implementar `SortedTableMap` con búsqueda binaria O(log n)
- Usar las operaciones adicionales de mapas ordenados (`find_range`, `find_lt`, etc.)
- Describir el funcionamiento de Skip Lists O(log n) esperado
- Aplicar `set`, `Counter` y `defaultdict` de Python para problemas reales

### 📚 Contenido

📚 *Data Structures and Algorithms in Python* — Goodrich, Tamassia, Goldwasser (Cap. 10 Secc. 3, 4, y 5)

---

## 🔍 Mapas Ordenados (Sorted Maps)

Un **mapa ordenado** mantiene las entradas **ordenadas por clave** y ofrece
operaciones adicionales que aprovechan ese orden.

### Operaciones Adicionales de SortedMap

| Operación | Descripción |
|-----------|-------------|
| `M.find_min()` | Entrada de clave mínima, o `None` |
| `M.find_max()` | Entrada de clave máxima, o `None` |
| `M.find_lt(k)` | Entrada de clave mayor más próxima menor que `k` |
| `M.find_le(k)` | Entrada de clave menor o igual a `k` |
| `M.find_gt(k)` | Entrada de clave menor más próxima mayor que `k` |
| `M.find_ge(k)` | Entrada de clave mayor o igual a `k` |
| `M.find_range(start, stop)` | Itera sobre entradas con `start ≤ k < stop` |
| `iter(M)` | Itera sobre claves en orden **ascendente** |
| `reversed(M)` | Itera sobre claves en orden **descendente** |

### Implementación: SortedTableMap

Internamente usa un **array ordenado** de `_Item`.
- Búsqueda: **búsqueda binaria** O(log n)
- Inserción/Eliminación: O(n) (desplazar elementos)
- Iteración: O(n) en orden

> **Trade-off**: ideal cuando se hacen muchas búsquedas de rango y pocas modificaciones.


In [ ]:
# ============================================================
# SortedTableMap: Búsqueda Binaria O(log n) (Código 10.3 del libro)
# ============================================================
from collections.abc import MutableMapping

class MapBase(MutableMapping):
    class _Item:
        __slots__ = '_key', '_value'
        def __init__(self, k, v): self._key = k; self._value = v
        def __eq__(self, other): return self._key == other._key
        def __ne__(self, other): return not (self == other)
        def __lt__(self, other): return self._key < other._key

class SortedTableMap(MapBase):
    """Implementación de Sorted Map usando array ordenado.
    
    Complejidad:
    - __getitem__:    O(log n)  — búsqueda binaria
    - __setitem__:    O(n)      — desplazar al insertar
    - __delitem__:    O(n)      — desplazar al eliminar
    - find_range:     O(s + log n)  donde s = tamaño del rango
    - __len__:        O(1)
    - __iter__:       O(n)
    """
    
    # ---- búsqueda binaria (privada) ----
    def _find_index(self, k, low, high):
        """Retorna índice en [low, high+1] tal que self._table[index]._key >= k.
        
        Si k existe: retorna su índice.
        Si k no existe: retorna el índice donde debería insertarse.
        """
        if high < low:
            return high + 1      # ningún elemento en rango
        else:
            mid = (low + high) // 2
            if k == self._table[mid]._key:
                return mid                              # encontrado exactamente
            elif k < self._table[mid]._key:
                return self._find_index(k, low, mid-1) # buscar en mitad izquierda
            else:
                return self._find_index(k, mid+1, high)# buscar en mitad derecha
    
    def __init__(self):
        self._table = []    # lista de _Item ordenada por clave
    
    def __len__(self):
        return len(self._table)
    
    def __getitem__(self, k):
        j = self._find_index(k, 0, len(self._table) - 1)
        if j < len(self._table) and self._table[j]._key == k:
            return self._table[j]._value
        raise KeyError(repr(k))
    
    def __setitem__(self, k, v):
        j = self._find_index(k, 0, len(self._table) - 1)
        if j < len(self._table) and self._table[j]._key == k:
            self._table[j]._value = v           # actualizar existente
        else:
            self._table.insert(j, self._Item(k, v))  # insertar nuevo
    
    def __delitem__(self, k):
        j = self._find_index(k, 0, len(self._table) - 1)
        if j == len(self._table) or self._table[j]._key != k:
            raise KeyError(repr(k))
        self._table.pop(j)
    
    def __iter__(self):
        for item in self._table:
            yield item._key
    
    def __reversed__(self):
        for item in reversed(self._table):
            yield item._key
    
    # ---- Operaciones adicionales de mapa ordenado ----
    def find_min(self):
        if len(self._table) == 0: return None
        item = self._table[0]
        return (item._key, item._value)
    
    def find_max(self):
        if len(self._table) == 0: return None
        item = self._table[-1]
        return (item._key, item._value)
    
    def find_ge(self, k):
        """Retorna (key, value) con la menor clave >= k."""
        j = self._find_index(k, 0, len(self._table) - 1)
        if j < len(self._table):
            return (self._table[j]._key, self._table[j]._value)
        return None
    
    def find_lt(self, k):
        """Retorna (key, value) con la mayor clave < k."""
        j = self._find_index(k, 0, len(self._table) - 1)
        if j > 0:
            item = self._table[j-1]
            return (item._key, item._value)
        return None
    
    def find_gt(self, k):
        """Retorna (key, value) con la menor clave > k."""
        j = self._find_index(k, 0, len(self._table) - 1)
        if j < len(self._table) and self._table[j]._key == k:
            j += 1   # saltar la clave exacta
        if j < len(self._table):
            return (self._table[j]._key, self._table[j]._value)
        return None
    
    def find_le(self, k):
        """Retorna (key, value) con la mayor clave <= k."""
        j = self._find_index(k, 0, len(self._table) - 1)
        if j < len(self._table) and self._table[j]._key == k:
            item = self._table[j]
            return (item._key, item._value)
        elif j > 0:
            item = self._table[j-1]
            return (item._key, item._value)
        return None
    
    def find_range(self, start, stop):
        """Itera sobre pares (k, v) con start <= k < stop.
        Si start=None, comienza desde el mínimo.
        Si stop=None, va hasta el máximo.
        """
        if start is None:
            j = 0
        else:
            j = self._find_index(start, 0, len(self._table) - 1)
        while j < len(self._table):
            if stop is not None and self._table[j]._key >= stop:
                break
            yield (self._table[j]._key, self._table[j]._value)
            j += 1
    
    def __repr__(self):
        items = ', '.join(f'{item._key!r}: {item._value!r}' for item in self._table)
        return 'SortedMap{' + items + '}'


# ============================================================
# Demo SortedTableMap
# ============================================================
print("=== SortedTableMap: Operaciones Básicas ===")
m = SortedTableMap()

entradas = [(2, 'B'), (4, 'D'), (6, 'F'), (8, 'H'), (10, 'J'),
            (1, 'A'), (3, 'C'), (5, 'E'), (7, 'G'), (9, 'I')]
for k, v in entradas:
    m[k] = v

print(f"Después de {len(entradas)} inserciones:")
print(f"  {m}")
print(f"  find_min()     = {m.find_min()}")
print(f"  find_max()     = {m.find_max()}")
print(f"  find_ge(4)     = {m.find_ge(4)}")
print(f"  find_ge(4.5)   = {m.find_ge(4.5)}")
print(f"  find_lt(5)     = {m.find_lt(5)}")
print(f"  find_gt(5)     = {m.find_gt(5)}")
print(f"  find_le(5)     = {m.find_le(5)}")
print(f"  find_le(5.5)   = {m.find_le(5.5)}")

print(f"\nfind_range(3, 8): {list(m.find_range(3, 8))}")
print(f"find_range(None, 4): {list(m.find_range(None, 4))}")
print(f"find_range(7, None): {list(m.find_range(7, None))}")

print(f"\nIteración ascendente:  {list(m)}")
print(f"Iteración descendente: {list(reversed(m))}")


## 📊 Aplicación: Base de Datos de Vuelos

Una aplicación clásica de `SortedMap` con `find_range` es una **base de datos de vuelos**
donde se consultan vuelos por rango de horas de salida.


In [ ]:
# ============================================================
# Aplicación: Base de datos de vuelos con SortedTableMap
# Ejemplo de la Sección 10.3.2 del libro
# ============================================================

# Base de datos de vuelos: hora_salida → (destino, precio)
vuelos = SortedTableMap()
vuelos['06:00'] = ('Buenos Aires', 12500)
vuelos['07:30'] = ('Córdoba', 8200)
vuelos['09:00'] = ('Mendoza', 9800)
vuelos['11:00'] = ('Bariloche', 15300)
vuelos['13:30'] = ('Salta', 11200)
vuelos['15:00'] = ('Rosario', 7500)
vuelos['17:00'] = ('Tucumán', 10100)
vuelos['19:30'] = ('Mar del Plata', 8800)
vuelos['22:00'] = ('Buenos Aires', 11000)

print("=== Base de Datos de Vuelos ===")
print("Vuelos de mañana (06:00 - 12:00):")
for hora, (dest, precio) in vuelos.find_range('06:00', '12:00'):
    print(f"  {hora}  →  {dest:20s}  ${precio:,}")

print()
print("Primer vuelo a las 10:00 o después:")
result = vuelos.find_ge('10:00')
if result:
    hora, (dest, precio) = result
    print(f"  {hora} → {dest} ${precio:,}")

print()
print("Último vuelo antes de las 14:00:")
result = vuelos.find_lt('14:00')
if result:
    hora, (dest, precio) = result
    print(f"  {hora} → {dest} ${precio:,}")

# ============================================================
# Skip Lists: Concepto Probabilístico (Sección 10.4)
# ============================================================
print()
print("="*50)
print("=== Skip Lists: Concepto (Sección 10.4) ===")
print()
print("""
Una Skip List es una estructura probabilística que generaliza
una lista enlazada ordenada con múltiples niveles de 'atajos'.

Nivel 3: ──────1─────────────────────────────────50──────────
Nivel 2: ──────1──────────17─────────────────────50──────────
Nivel 1: ──────1──────────17──────────31──────────50──────────
Nivel 0: ──────1────9─────17────26────31────38────50────55────

Cada elemento se inserta en el nivel 0. Con probabilidad p=1/2
se 'sube' al nivel 1, con probabilidad p²=1/4 al nivel 2, etc.

Complejidad ESPERADA (análisis probabilístico):
  - Búsqueda:   O(log n)
  - Inserción:  O(log n)  
  - Eliminación: O(log n)
  - Espacio:    O(n) esperado

Ventajas:
  ✅ Simple de implementar
  ✅ Sin necesidad de rebalanceo (a diferencia de árboles AVL/RB)
  ✅ Buen rendimiento en la práctica
  
Usos reales:
  🔥 Redis: SORTED SET usa skip list internamente
  🔥 LevelDB/RocksDB: memtable usa skip list
  🔥 Java: ConcurrentSkipListMap en java.util.concurrent
""")


## 🗂️ Sets, Multisets y Multimaps (Sección 10.5)

Python ofrece implementaciones nativas de estas estructuras.

### `set` y `frozenset` de Python

| Operación | Descripción | Complejidad |
|-----------|-------------|-------------|
| `s.add(e)` | Agrega elemento | O(1) esperado |
| `e in s` | Pertenencia | O(1) esperado |
| `s.discard(e)` | Elimina si existe | O(1) esperado |
| `s | t` | Unión | O(len(s) + len(t)) |
| `s & t` | Intersección | O(min(len(s), len(t))) |
| `s - t` | Diferencia | O(len(s)) |
| `s ^ t` | Diferencia simétrica | O(len(s) + len(t)) |
| `s <= t` | Subconjunto | O(len(s)) |

### `Counter` como Multiset

`Counter` del módulo `collections` es un multiset: permite múltiples ocurrencias del mismo elemento.

### `defaultdict` como Multimap

`defaultdict(list)` implementa un multimap: múltiples valores para la misma clave.


In [ ]:
# ============================================================
# Sets, Counter y defaultdict de Python (Sección 10.5)
# ============================================================
from collections import Counter, defaultdict

# ---- SET: operaciones fundamentales ----
print("=== Set Operations ===")
A = {1, 2, 3, 4, 5}
B = {3, 4, 5, 6, 7}

print(f"A = {A}")
print(f"B = {B}")
print(f"A | B (unión)            = {A | B}")
print(f"A & B (intersección)     = {A & B}")
print(f"A - B (diferencia A-B)   = {A - B}")
print(f"B - A (diferencia B-A)   = {B - A}")
print(f"A ^ B (dif. simétrica)   = {A ^ B}")
print(f"A <= B (A subconj de B?) = {A <= B}")
print("{3,4} <= A             =", frozenset([3,4]) <= A)

# ---- COUNTER como MULTISET ----
print()
print("=== Counter (Multiset) ===")
texto = "el gato come el pato que el gato caza"
c = Counter(texto.split())
print(f"Texto: '{texto}'")
print(f"Counter: {c}")
print(f"Más comunes: {c.most_common(3)}")

# Operaciones aritméticas de Counter
c1 = Counter({'a': 3, 'b': 2, 'c': 1})
c2 = Counter({'a': 1, 'b': 2, 'd': 3})
print(f"\nc1 = {c1}")
print(f"c2 = {c2}")
print(f"c1 + c2 = {c1 + c2}")  # suma de conteos
print(f"c1 - c2 = {c1 - c2}")  # diferencia (elimina ≤ 0)
print(f"c1 & c2 = {c1 & c2}")  # mínimo
print(f"c1 | c2 = {c1 | c2}")  # máximo

# ---- DEFAULTDICT como MULTIMAP ----
print()
print("=== defaultdict (Multimap) ===")
# Agrupar estudiantes por nota
notas = [('Ana', 8), ('Luis', 9), ('María', 8), ('Juan', 7),
         ('Pedro', 9), ('Laura', 7), ('Carlos', 10)]

por_nota = defaultdict(list)
for nombre, nota in notas:
    por_nota[nota].append(nombre)

print("Estudiantes agrupados por nota:")
for nota in sorted(por_nota.keys(), reverse=True):
    print(f"  Nota {nota}: {sorted(por_nota[nota])}")

# ---- FROZENSET como clave de diccionario ----
print()
print("=== frozenset como clave ===")
# Contar pares de colegas (sin importar orden)
reuniones = [
    frozenset(['Ana', 'Luis']),
    frozenset(['Luis', 'María']),
    frozenset(['Ana', 'Luis']),
    frozenset(['María', 'Ana']),
]
freq_reuniones = Counter(reuniones)
print("Frecuencia de reuniones entre pares:")
for par, count in freq_reuniones.most_common():
    nombres = sorted(par)
    print(f"  {nombres[0]} ↔ {nombres[1]}: {count} veces")


## 🌟 Resumen del Capítulo 10

### Tabla Comparativa de Implementaciones de Mapa

| Estructura | `get` | `set` | `del` | `iter` | Ordenado | Notas |
|-----------|-------|-------|-------|--------|----------|-------|
| UnsortedTableMap | O(n) | O(n) | O(n) | O(n) | No | Base pedagógica |
| ChainHashMap | O(1)* | O(1)* | O(1)* | O(n) | No | *esperado |
| ProbeHashMap | O(1)* | O(1)* | O(1)* | O(n) | No | *esperado, λ<0.5 |
| SortedTableMap | O(log n) | O(n) | O(n) | O(n) | ✅ | find_range O(s+log n) |
| Skip List | O(log n)* | O(log n)* | O(log n)* | O(n) | ✅ | *esperado |
| `dict` Python | O(1)* | O(1)* | O(1)* | O(n) | desde 3.7 ins-ord | Más rápido |

**λ = factor de carga = n / capacidad_tabla**

### ¿Cuándo usar cada estructura?

```
¿Necesitas operaciones de rango (find_lt, find_range)?
  → SortedTableMap / Skip List

¿Las claves son hasheables y solo necesitas get/set/del?
  → dict (Python built-in) o ChainHashMap/ProbeHashMap

¿Necesitas multiples valores por clave?
  → defaultdict(list)

¿Necesitas contar frecuencias?
  → Counter

¿Necesitas operaciones de conjuntos?
  → set / frozenset
```


---

🔗 **Referencia:** Goodrich et al., Cap. 10.3-5

---

![Creative Commons](https://mirrors.creativecommons.org/presskit/buttons/80x15/png/by-sa.png)

© 2026 Cátedra Programación III – Lic. Sistemas – FCAD/UNER

This notebook is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0)

<https://creativecommons.org/licenses/by-sa/4.0/>